# 🔍 RAG + Vector Databases

## Build Smarter AI Apps with LangChain

Welcome to the live session on Retrieval Augmented Generation (RAG) and Vector Databases! In this notebook, you'll learn how to build production-ready RAG systems that ground LLM responses in your own data.

---

## 🚀 Getting Started

### Prerequisites
- Python 3.8 or higher
- Jupyter Notebook installed
- OpenAI API key (for embeddings and LLM calls)

### Setup Instructions

**Option 1: Using requirements.txt (Recommended)**
```bash
pip install -r requirements.txt
```

**Option 2: Manual Installation**
```bash
pip install langchain langchain-openai langchain-community langchain-text-splitters chromadb pypdf python-dotenv
```

### Environment Setup

Create a `.env` file in this directory with your OpenAI API key:
```
OPENAI_API_KEY=your_api_key_here
```

---

## 📚 What You'll Learn

1. **Why RAG Exists** — Understanding LLM limitations and how RAG solves them
2. **RAG Architecture** — The complete flow from indexing to query
3. **Embeddings Deep Dive** — How text becomes vectors and why it matters
4. **Chunking Strategies** — The make-or-break step most people get wrong
5. **Vector Databases** — Storing and searching semantic data at scale
6. **Live Build** — Build a working Document Q&A system
7. **Evaluation** — How to measure and improve your RAG system

### 🎯 By the End of This Notebook

- ✅ Understand the complete RAG architecture
- ✅ Master embeddings and vector similarity search
- ✅ Build a production-ready Document Q&A system
- ✅ Know how to evaluate and debug RAG systems
- ✅ Be ready to build RAG applications on your own data

---

## 🔑 Key Concepts

| Concept | What It Is | Why It Matters |
|---------|------------|----------------|
| **RAG** | Retrieval + Augmentation + Generation | Grounds LLM responses in your data |
| **Embeddings** | Vector representations of text | Enables semantic search |
| **Vector DB** | Database optimized for high-dimensional vectors | Fast similarity search at scale |
| **Chunking** | Splitting documents into smaller pieces | Critical for retrieval quality |
| **Retrieval** | Finding relevant context for queries | Determines answer quality |

---

## 📖 How to Use This Notebook

1. **Run cells in order** - Each section builds on the previous one
2. **Read the markdown cells** - They contain important explanations
3. **Experiment** - Try modifying parameters to see what happens
4. **Test with your data** - Replace example documents with your own

**Ready? Let's start by installing dependencies!**


In [1]:
# Install all required packages:
%pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters langchain-chroma chromadb pypdf python-dotenv ragas -q

# Verify installation
import sys
print(f"Python version: {sys.version}")

# Verify key packages are installed
try:
    import langchain
    import langchain_core
    print(f"✅ LangChain version: {langchain.__version__}")
    print("✅ All core packages installed successfully!")
except ImportError as e:
    print(f"⚠️  Error: {e}")
    print("Please ensure all packages are installed correctly.")



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Python version: 3.14.2 (main, Dec  5 2025, 16:49:16) [Clang 17.0.0 (clang-1700.6.3.2)]
✅ LangChain version: 1.3.12
✅ All core packages installed successfully!


---

## 🛠️ Setup & Imports

Let's import all the essential components we'll need throughout this notebook.

In [ ]:
# Standard library imports
import os
from pathlib import Path
from dotenv import load_dotenv

# LangChain core imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# Modern LangChain agent imports (latest pattern from LangChain docs)
from langchain.agents import create_agent
from langchain.tools import tool

# Load environment variables - check current directory first, then parent
current_dir = Path.cwd()
env_path = current_dir / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loading .env from: {env_path}")
else:
    # Try parent directory
    parent_env = current_dir.parent / '.env'
    if parent_env.exists():
        load_dotenv(parent_env)
        print(f"✅ Loading .env from: {parent_env}")
    else:
        # Fallback to default behavior (searches current and parent directories)
        load_dotenv()
        print("⚠️  No .env file found. Using default load_dotenv() search")

# Verify API key is set
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("⚠️  WARNING: OPENAI_API_KEY not found in environment variables!")
    print("Please create a .env file in this directory with your OpenAI API key:")
    print(f"   {current_dir / '.env'}")
    print("\nFormat: OPENAI_API_KEY=sk-...")
else:
    # Validate API key format (basic check)
    if not api_key.startswith('sk-'):
        print("⚠️  WARNING: API key format looks incorrect. Should start with 'sk-'")
    else:
        print("✅ OpenAI API key loaded successfully!")
        # Test the API key by making a simple call
        try:
            test_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=5)
            test_llm.invoke("Hi")
            print("✅ API key validated successfully!")
        except Exception as e:
            print(f"❌ ERROR: API key validation failed!")
            print(f"   Error: {str(e)}")
            print("\n💡 Please check:")
            print("   1. Your API key is correct (get it from https://platform.openai.com/api-keys)")
            print("   2. You have sufficient credits in your OpenAI account")
            print("   3. The .env file is in the correct location")
            raise

print("\n✅ All imports successful!")
print("📚 Using modern LangChain patterns: create_agent with @tool decorator")


/var/folders/4z/ycq9kt293wq0pq0hwvd979gr0000gn/T/ipykernel_339/4203647281.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


✅ Loading .env from: /Users/akshika47/Documents/GitHub/AI-Internship/ai-engineering-bootcamp-v2/week-2/rag-vector-databases/.env
✅ OpenAI API key loaded successfully!


---

## 🎯 Part 1: Why RAG Exists

### The Problem with LLMs

LLMs are incredibly powerful, but they have critical limitations:

1. **Knowledge Cutoff** - They only know what they were trained on, up to a fixed training cutoff date
2. **Hallucination** - They can confidently make up information
3. **No Access to Private Data** - They can't access your documents, databases, or internal knowledge
4. **Static Knowledge** - They can't learn new information after training

### How RAG Solves This

**RAG = Retrieval + Augmentation + Generation**

1. **Retrieval**: Find relevant information from your knowledge base
2. **Augmentation**: Add that information to the LLM's prompt
3. **Generation**: LLM generates an answer grounded in the retrieved context

This gives you:
- ✅ Up-to-date information
- ✅ Access to private data
- ✅ Reduced hallucination
- ✅ Source attribution

### RAG vs Fine-Tuning vs Prompt Engineering

| Approach | What It Does | Best For |
|----------|-------------|----------|
| **RAG** | Adds knowledge to prompts | New information, private data, facts |
| **Fine-tuning** | Changes model behavior | Style, format, domain-specific tasks |
| **Prompt Engineering** | Optimizes prompts | Simple tasks, no code changes needed |

**Key insight:** RAG handles *knowledge*, fine-tuning handles *behavior* — they're not either/or!


---

## 🏗️ Part 2: The RAG Architecture

The RAG architecture has two main phases:

### Phase 1: Indexing (One-time setup)

1. **Load your knowledge base** - Documents, PDFs, text files, etc.
2. **Split into chunks** - Break documents into manageable pieces
3. **Convert to vectors** - Use an embedding model to create vector representations
4. **Store in Vector DB** - Save vectors for fast retrieval

### Phase 2: Query (Real-time)

1. **User asks a question** - "What are the key findings?"
2. **Embed the query** - Convert the question to a vector
3. **Find similar chunks** - Search the vector database for relevant documents
4. **Add context to prompt** - Combine retrieved chunks with the original question
5. **LLM generates answer** - The model uses the context to answer
6. **Return grounded response** - Answer with source attribution

### Visual Flow

```
INDEXING:
Knowledge Base → Chunks → Embedding Model → Vector DB

QUERY:
User Query → Embedding Model → Vector DB (search) → Top K Docs → 
Augment Prompt → LLM → Output
```

Let's build this step by step!


---

## 🔢 Part 3: Embeddings Deep Dive

### What are Embeddings?

Embeddings are **vector representations of text** that capture semantic meaning. Similar texts have similar vectors.

### Key Properties

1. **Semantic Similarity** - "dog" and "puppy" are close in vector space
2. **Fixed Dimensions** - Each embedding has a fixed size (e.g., 1536 for OpenAI's text-embedding-3-small)
3. **Distance = Meaning** - Closer vectors = more similar meaning

### Why Embeddings Matter

- **Traditional search**: "Python programming" won't match "coding in Python"
- **Semantic search**: "Python programming" WILL match "coding in Python" because they're semantically similar

Let's see embeddings in action:


In [ ]:
# Initialize the embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Example texts to embed
texts = [
    "Python is a programming language",
    "Coding in Python is fun",
    "Dogs are loyal pets",
    "Cats are independent animals"
]

# Generate embeddings
text_embeddings = embeddings.embed_documents(texts)

print(f"Number of texts: {len(texts)}")
print(f"Embedding dimensions: {len(text_embeddings[0])}")
print(f"\nFirst embedding (first 5 values): {text_embeddings[0][:5]}")

import numpy as np

def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors"""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Full similarity matrix: every text compared to every other text
print("\n📊 Cosine similarity matrix (1.00 = identical direction in vector space):\n")
print(" " * 42 + "".join(f"  [{j}] " for j in range(len(texts))))
for i, emb_i in enumerate(text_embeddings):
    row = "".join(f" {cosine_similarity(emb_i, emb_j):.2f} " for emb_j in text_embeddings)
    print(f"[{i}] {texts[i]:<38}{row}")

print("\n✅ Read the matrix:")
print("   • The two Python sentences are close (~0.6)")
print("   • Python vs pets is far apart (~0.1)")
print("   • Dogs vs cats sit in between — different animals, same topic!")


Number of texts: 4
Embedding dimensions: 1536

First embedding (first 5 values): [-0.01096343994140625, -0.0204010009765625, 0.0188140869140625, -0.002838134765625, 0.0156402587890625]

📊 Cosine similarity matrix (1.00 = identical direction in vector space):

                                            [0]   [1]   [2]   [3] 
[0] Python is a programming language       1.00  0.59  0.13  0.13 
[1] Coding in Python is fun                0.59  1.00  0.09  0.12 
[2] Dogs are loyal pets                    0.13  0.09  1.00  0.48 
[3] Cats are independent animals           0.13  0.12  0.48  1.00 

✅ Read the matrix:
   • The two Python sentences are close (~0.6)
   • Python vs pets is far apart (~0.1)
   • Dogs vs cats sit in between — different animals, same topic!


---

## ✂️ Part 4: Chunking Strategies

### Why Chunking Matters

**Chunking is the make-or-break step in RAG.** Bad chunking = bad retrieval = bad answers.

### The Challenge

- **Too small**: Lose context, incomplete information
- **Too large**: Noise, irrelevant information, token limits
- **Wrong boundaries**: Split sentences, lose meaning

### Chunking Strategies

| Strategy | How It Works | Best For |
|----------|-------------|----------|
| **Fixed-size** | Split every N characters/tokens | Simple, predictable |
| **Recursive** | Split by paragraphs → sentences → words | General-purpose (LangChain default) |
| **Semantic** | Split when meaning shifts | High-quality but slower |
| **Document-aware** | Split by headers, sections, pages | Structured docs (markdown, HTML) |

### Best Practices

1. **Use overlap** - 10-20% overlap prevents losing context at boundaries
2. **Respect structure** - Don't split in the middle of sentences
3. **Consider your use case** - Technical docs need larger chunks, Q&A needs smaller

Instead of just claiming these things, we'll **prove each one** with four demos:

1. **Demo 1 — Fixed-size vs recursive**: watch fixed-size splitting chop words in half, and recursive splitting fall back from paragraphs → sentences → words
2. **Demo 2 — Overlap made visible**: see the *exact text* that gets repeated between consecutive chunks
3. **Demo 3 — Document-aware chunking**: split markdown by its headers, keeping the section title as metadata
4. **Demo 4 — Chunk size changes answers**: the *same question* retrieves complete or broken context depending only on chunk size


In [ ]:
# ============ Demo 1: Fixed-size vs Recursive splitting ============
from langchain_text_splitters import CharacterTextSplitter, MarkdownHeaderTextSplitter

# A realistic sample document: markdown headers, short paragraphs, AND one
# long paragraph (~750 chars) that cannot fit in a single 300-char chunk —
# that's what forces the recursive splitter to show its fallback behavior.
chunking_doc = """# Understanding Retrieval Augmented Generation

## Why LLMs Need External Knowledge

Large Language Models are trained on a fixed snapshot of the internet. The moment training ends, the model starts falling out of date, and it has no idea what is inside your company's private documents.

## How Retrieval Fixes This

Retrieval Augmented Generation solves the knowledge problem in a surprisingly direct way, and it is worth walking through slowly because every design decision in a RAG system flows from this one idea. When a user asks a question, the system does not immediately send that question to the language model. Instead, it first searches a database of your own documents for passages that look relevant to the question. Those passages are then pasted into the prompt, right next to the user's question, together with an instruction that says answer using only the context provided. The language model now behaves less like an oracle and more like a very fast reading assistant that composes an answer grounded in the retrieved passages.

## Benefits of RAG

RAG systems offer four major benefits. First, they reduce hallucination, because every answer is grounded in retrieved source text. Second, they give the model access to up-to-date information published after its training cutoff. Third, they unlock private and internal documents that a public model has never seen. Fourth, they enable source attribution, so every claim in an answer can be traced back to the exact document it came from.

## A Note on Components

Practitioners describe the retriever, the vector index, the embedding model, the chunking pipeline, and the generator as separate components, even though in a small system they may all live inside a single short Python file.
"""

print(f"📄 Document length: {len(chunking_doc)} characters")


def show_boundaries(chunks, n_pairs=4):
    """Print where each cut lands: end of one chunk ✂️ start of the next."""
    for i in range(min(n_pairs, len(chunks) - 1)):
        end = chunks[i][-45:].replace("\n", "⏎")
        start = chunks[i + 1][:45].replace("\n", "⏎")
        print(f"   …{end} ✂️ {start}…")


# --- Strategy 1: TRUE fixed-size splitting (cut every 300 chars, no matter what) ---
fixed_splitter = CharacterTextSplitter(separator="", chunk_size=300, chunk_overlap=0)
chunks_fixed = fixed_splitter.split_text(chunking_doc)

print(f"\n🔪 FIXED-SIZE (300 chars) → {len(chunks_fixed)} chunks. Where the cuts land:")
show_boundaries(chunks_fixed)
print("   ⚠️  Cuts land mid-word and mid-sentence — meaning is destroyed at every boundary.")

# --- Strategy 2: RECURSIVE splitting (try \n\n, then \n, then '. ', then ' ') ---
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=0,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks_recursive = recursive_splitter.split_text(chunking_doc)

print(f"\n🧠 RECURSIVE (300 chars) → {len(chunks_recursive)} chunks. Where the cuts land:")
show_boundaries(chunks_recursive)
print("   ✅ Cuts land at paragraph and sentence boundaries — each chunk stays meaningful.")
print("   💡 The long 'How Retrieval Fixes This' paragraph didn't fit in 300 chars,")
print("      so the splitter FELL BACK from paragraph-splits to sentence-splits there.")


📄 Document length: 1759 characters

🔪 FIXED-SIZE (300 chars) → 6 chunks. Where the cuts land:
   …our company's private documents.⏎⏎## How Retr ✂️ ieval Fixes This⏎⏎Retrieval Augmented Generat…
   …tem does not immediately send that question t ✂️ o the language model. Instead, it first searc…
   …swer using only the context provided. The lan ✂️ guage model now behaves less like an oracle a…
   … answer is grounded in retrieved source text. ✂️ Second, they give the model access to up-to-d…
   ⚠️  Cuts land mid-word and mid-sentence — meaning is destroyed at every boundary.

🧠 RECURSIVE (300 chars) → 10 chunks. Where the cuts land:
   …t is inside your company's private documents. ✂️ ## How Retrieval Fixes This…
   …## How Retrieval Fixes This ✂️ Retrieval Augmented Generation solves the kno…
   …sion in a RAG system flows from this one idea ✂️ . When a user asks a question, the system doe…
   …r passages that look relevant to the question ✂️ . Those passages are then pasted into the pro

In [ ]:
# ============ Demo 2: Overlap made visible ============
# Overlap repeats a little text between consecutive chunks so that a fact
# sitting near a boundary isn't orphaned. Let's SEE the repeated text.
#
# We take the long paragraph from our document and split it at word level —
# which is what the recursive splitter does to any text with no structure
# (no paragraph breaks, no sentence that fits) left to cut at.

long_paragraph = (
    chunking_doc.split("## How Retrieval Fixes This")[1]
    .split("## Benefits of RAG")[0]
    .strip()
)

def find_overlap(chunk_a, chunk_b):
    """Return the longest text that ends chunk_a AND starts chunk_b."""
    for size in range(min(len(chunk_a), len(chunk_b)), 0, -1):
        if chunk_a[-size:] == chunk_b[:size]:
            return chunk_a[-size:]
    return ""

for overlap in [0, 40]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=150, chunk_overlap=overlap, separators=[" ", ""],
    )
    demo_chunks = splitter.split_text(long_paragraph)
    print("=" * 70)
    print(f"chunk_size=150, chunk_overlap={overlap} → {len(demo_chunks)} chunks")
    print("=" * 70)
    for i in range(3):
        shared = find_overlap(demo_chunks[i], demo_chunks[i + 1])
        print(f"Chunk {i + 1} ends:   …{demo_chunks[i][-45:]!r}")
        print(f"Chunk {i + 2} starts: {demo_chunks[i + 1][:45]!r}…")
        if shared:
            print(f"   🔁 shared text ({len(shared)} chars): {shared!r}")
        else:
            print("   ❌ no shared text — a fact near this cut is split in half")
        print()

print("💡 With overlap=0, a fact near a cut is split so that NEITHER chunk holds")
print("   it completely. With overlap=40, the words around every cut appear in")
print("   BOTH chunks — whichever chunk retrieval picks, the boundary context")
print("   comes with it. (When chunks end at natural paragraph boundaries, the")
print("   recursive splitter adds no overlap — it's insurance for awkward cuts.)")


chunk_size=150, chunk_overlap=0 → 5 chunks
Chunk 1 ends:   …'h walking through slowly because every design'
Chunk 2 starts: 'decision in a RAG system flows from this one '…
   ❌ no shared text — a fact near this cut is split in half

Chunk 2 ends:   …'mmediately send that question to the language'
Chunk 3 starts: 'model. Instead, it first searches a database '…
   ❌ no shared text — a fact near this cut is split in half

Chunk 3 ends:   …' the question. Those passages are then pasted'
Chunk 4 starts: "into the prompt, right next to the user's que"…
   ❌ no shared text — a fact near this cut is split in half

chunk_size=150, chunk_overlap=40 → 7 chunks
Chunk 1 ends:   …'h walking through slowly because every design'
Chunk 2 starts: 'through slowly because every design decision '…
   🔁 shared text (35 chars): 'through slowly because every design'

Chunk 2 ends:   …'s a question, the system does not immediately'
Chunk 3 starts: 'the system does not immediately send that que'…
   🔁 shared 

In [ ]:
# ============ Demo 3: Document-aware chunking (MarkdownHeaderTextSplitter) ============
# Our document is markdown — so split it BY ITS HEADERS. Each chunk is one
# coherent section, and the section title travels along as searchable metadata.

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "title"), ("##", "section")]
)
md_chunks = md_splitter.split_text(chunking_doc)

print(f"📑 Markdown-aware splitting → {len(md_chunks)} chunks (one per section):\n")
for i, doc in enumerate(md_chunks, 1):
    print(f"Chunk {i} — metadata: {doc.metadata}")
    print(f"   {doc.page_content[:90].replace(chr(10), ' ')}…\n")

print("💡 Why this is powerful: the metadata lets you FILTER retrieval")
print("   (e.g. only search chunks where section == 'Benefits of RAG'),")
print("   and chunk boundaries always match how the author organized ideas.")


📑 Markdown-aware splitting → 4 chunks (one per section):

Chunk 1 — metadata: {'title': 'Understanding Retrieval Augmented Generation', 'section': 'Why LLMs Need External Knowledge'}
   Large Language Models are trained on a fixed snapshot of the internet. The moment training…

Chunk 2 — metadata: {'title': 'Understanding Retrieval Augmented Generation', 'section': 'How Retrieval Fixes This'}
   Retrieval Augmented Generation solves the knowledge problem in a surprisingly direct way, …

Chunk 3 — metadata: {'title': 'Understanding Retrieval Augmented Generation', 'section': 'Benefits of RAG'}
   RAG systems offer four major benefits. First, they reduce hallucination, because every ans…

Chunk 4 — metadata: {'title': 'Understanding Retrieval Augmented Generation', 'section': 'A Note on Components'}
   Practitioners describe the retriever, the vector index, the embedding model, the chunking …

💡 Why this is powerful: the metadata lets you FILTER retrieval
   (e.g. only search chunks wher

In [ ]:
# ============ Demo 4: Chunk size changes the ANSWER ============
# The payoff: the SAME question against the SAME document retrieves complete
# or broken context depending only on chunk size.

question = "What are the benefits of RAG?"

for label, size in [("🔴 TOO-SMALL chunks (size=100)", 100),
                    ("🟢 RIGHT-SIZED chunks (size=500)", 500)]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=0,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    doc_chunks = splitter.split_text(chunking_doc)

    # In-memory store, deleted after use, so this cell is safe to re-run
    store = Chroma.from_texts(
        texts=doc_chunks, embedding=embeddings,
        collection_name=f"chunk_size_demo_{size}",
    )
    results = store.similarity_search(question, k=2)
    store.delete_collection()

    print(f"\n{'=' * 70}")
    print(f"{label} → document became {len(doc_chunks)} chunks")
    print(f"{'=' * 70}")
    print(f"🔍 Query: {question!r} — top-2 retrieved chunks:\n")
    for i, doc in enumerate(results, 1):
        print(f"  {i}. {doc.page_content.replace(chr(10), ' ')}\n")

print("💡 With 100-char chunks the 'four major benefits' got scattered across")
print("   separate chunks — retrieval finds a fragment, and the LLM would answer")
print("   from an incomplete list. With 500-char chunks the whole benefits")
print("   paragraph arrives intact. Same data, same query — chunking decided the answer.")



🔴 TOO-SMALL chunks (size=100) → document became 26 chunks
🔍 Query: 'What are the benefits of RAG?' — top-2 retrieved chunks:

  1. ## Benefits of RAG

  2. RAG systems offer four major benefits


🟢 RIGHT-SIZED chunks (size=500) → document became 5 chunks
🔍 Query: 'What are the benefits of RAG?' — top-2 retrieved chunks:

  1. ## Benefits of RAG  RAG systems offer four major benefits. First, they reduce hallucination, because every answer is grounded in retrieved source text. Second, they give the model access to up-to-date information published after its training cutoff. Third, they unlock private and internal documents that a public model has never seen. Fourth, they enable source attribution, so every claim in an answer can be traced back to the exact document it came from.  ## A Note on Components

  2. Retrieval Augmented Generation solves the knowledge problem in a surprisingly direct way, and it is worth walking through slowly because every design decision in a RAG system flows 

---

## 🗄️ Part 5: Vector Databases

### What is a Vector Database?

A **vector database** is specialized storage optimized for:
- High-dimensional vectors (hundreds to thousands of dimensions)
- Fast similarity search (finding nearest neighbors)
- Scalability (millions of vectors)

### Why Not Regular Databases?

Regular SQL databases are great for exact matches, but terrible for:
- "Find documents similar to this query"
- Semantic search
- High-dimensional data

### Vector Database Options

| Database | Best For | Notes |
|----------|----------|-------|
| **ChromaDB** | Development, small-medium scale | Easy to use, Python-native |
| **Pinecone** | Production, large scale | Managed service, fast |
| **Weaviate** | Enterprise, complex queries | Open source, feature-rich |
| **Qdrant** | Performance-critical | Fast, open source |
| **Milvus** | Very large scale | Distributed, enterprise |

For this session, we'll use **ChromaDB** - it's simple, works great for learning, and is perfect for production at small-medium scale.

### How Vector Search Works

1. Query is embedded → becomes a vector
2. Vector DB finds vectors with highest cosine similarity
3. Returns top K most similar documents

Let's set up ChromaDB:


In [ ]:
# Create a simple vector store with sample documents
sample_docs = [
    "Python is a high-level programming language known for its simplicity.",
    "Machine learning is a subset of artificial intelligence.",
    "Vector databases enable semantic search over large text collections.",
    "LangChain is a framework for building LLM applications.",
    "RAG combines retrieval and generation for grounded AI responses."
]

# Each sentence is already a good-sized chunk for this quick demo
# (chunking strategies were covered in Part 4).
#
# We keep this store IN MEMORY and reset the collection first, so
# re-running this cell can never create duplicate entries.
demo_vectorstore = Chroma(collection_name="quick_demo", embedding_function=embeddings)
demo_vectorstore.reset_collection()
demo_vectorstore.add_texts(sample_docs)

print(f"✅ Created in-memory vector store with {len(sample_docs)} documents")

# Test similarity search
query = "What is Python?"
results = demo_vectorstore.similarity_search(query, k=3)

print(f"\n🔍 Query: '{query}'")
print(f"\n📄 Top {len(results)} results:")
for i, doc in enumerate(results, 1):
    print(f"\n  {i}. {doc.page_content}")


✅ Created in-memory vector store with 5 documents

🔍 Query: 'What is Python?'

📄 Top 3 results:

  1. Python is a high-level programming language known for its simplicity.

  2. Machine learning is a subset of artificial intelligence.

  3. RAG combines retrieval and generation for grounded AI responses.


---

## 🛠️ Part 6: Live Build - Document Q&A System

Now let's build a complete RAG system step by step!

### What We're Building

A Document Q&A system that can:
1. Load PDF or text documents
2. Chunk and embed them
3. Store in a vector database
4. Answer questions based on the documents

### Architecture

```
PDF/Text Files → Load → Chunk → Embed → Store in ChromaDB
                                                      ↓
User Query → Embed → Search → Retrieve Top K → Augment Prompt → LLM → Answer
```

Let's build it!


### Step 1: Create Sample Document

First, let's create a sample document to work with. In production, you'd load your own PDFs or text files.


In [ ]:
# Create a sample document file for demonstration
sample_content = """
# Introduction to RAG Systems

## What is RAG?

Retrieval Augmented Generation (RAG) is a technique that enhances the capabilities of Large Language Models (LLMs) by providing them with access to external knowledge sources.

## How RAG Works

RAG systems work in two phases:

### Phase 1: Indexing
1. Documents are loaded from various sources (PDFs, databases, APIs)
2. Documents are split into smaller chunks
3. Each chunk is converted to a vector using an embedding model
4. Vectors are stored in a vector database

### Phase 2: Querying
1. User asks a question
2. Question is converted to a vector
3. Vector database finds similar document chunks
4. Retrieved chunks are added to the LLM prompt
5. LLM generates an answer based on the retrieved context

## Benefits of RAG

- Reduces hallucination by grounding answers in real data
- Allows access to up-to-date information
- Enables use of private/internal documents
- Provides source attribution for answers

## Best Practices

1. Use appropriate chunk sizes (typically 500-1000 characters)
2. Include overlap between chunks (10-20%)
3. Choose the right embedding model for your domain
4. Test retrieval quality before optimizing generation
5. Use metadata filtering when possible
"""

# Save to a text file
with open("sample_rag_document.txt", "w") as f:
    f.write(sample_content)

print("✅ Created sample document: sample_rag_document.txt")
print(f"📄 Document length: {len(sample_content)} characters")


✅ Created sample document: sample_rag_document.txt
📄 Document length: 1237 characters


### Step 2: Load Documents

LangChain supports 80+ document loaders for different formats.


In [ ]:
# Load the document
loader = TextLoader("sample_rag_document.txt")
documents = loader.load()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"📄 First document preview:")
print("-" * 60)
print(documents[0].page_content[:300] + "...")
print("-" * 60)
print(f"\n📊 Document metadata: {documents[0].metadata}")


✅ Loaded 1 document(s)
📄 First document preview:
------------------------------------------------------------

# Introduction to RAG Systems

## What is RAG?

Retrieval Augmented Generation (RAG) is a technique that enhances the capabilities of Large Language Models (LLMs) by providing them with access to external knowledge sources.

## How RAG Works

RAG systems work in two phases:

### Phase 1: Indexing
1...
------------------------------------------------------------

📊 Document metadata: {'source': 'sample_rag_document.txt'}


### Step 3: Chunk Documents

Split documents into manageable chunks with overlap.


In [ ]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Size of each chunk
    chunk_overlap=100,   # Overlap between chunks (important!)
    separators=["\n\n", "\n", ". ", " ", ""]  # Try to split at these boundaries
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"✅ Split document into {len(chunks)} chunks")
print(f"\n📊 Chunk statistics:")
print(f"  Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} characters")
print(f"  Min chunk size: {min(len(c.page_content) for c in chunks)} characters")
print(f"  Max chunk size: {max(len(c.page_content) for c in chunks)} characters")

print(f"\n📄 First chunk preview:")
print("-" * 60)
print(chunks[0].page_content)
print("-" * 60)


✅ Split document into 4 chunks

📊 Chunk statistics:
  Average chunk size: 325 characters
  Min chunk size: 271 characters
  Max chunk size: 463 characters

📄 First chunk preview:
------------------------------------------------------------
# Introduction to RAG Systems

## What is RAG?

Retrieval Augmented Generation (RAG) is a technique that enhances the capabilities of Large Language Models (LLMs) by providing them with access to external knowledge sources.

## How RAG Works

RAG systems work in two phases:
------------------------------------------------------------


### Step 4: Create Embeddings and Vector Store

Convert chunks to vectors and store in ChromaDB.

> ⚠️ **Common gotcha:** `Chroma.from_documents(...)` with a `persist_directory` **appends** to whatever is already on disk. Re-run the cell twice and every chunk is stored twice — retrieval then returns duplicates that crowd out the genuinely relevant chunks (and silently ruin answer quality). Always start from a clean directory, as below, or use stable document IDs.


In [ ]:
# Reuse the embeddings model initialized earlier (Part 3)
import gc
import shutil
from pathlib import Path

# Always build a fresh on-disk store (safe to re-run this cell).
PERSIST_DIR = (Path.cwd() / "rag_vector_db").resolve()

# Close any previous client before deleting files (avoids SQLite lock errors).
if "vectorstore" in globals():
    try:
        vectorstore._client.close()
    except Exception:
        pass
    del vectorstore
    gc.collect()

shutil.rmtree(PERSIST_DIR, ignore_errors=True)
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

# Create vector store from chunks
# This will:
# 1. Generate embeddings for all chunks
# 2. Store them in ChromaDB
# 3. Persist to disk for later use

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(PERSIST_DIR),
    collection_name="rag_documents",
)

print("✅ Created vector store!")
print(f"📁 Database saved to: {PERSIST_DIR}")
print(f"📊 Stored {len(chunks)} document chunks")

# Test retrieval
test_query = "What are the benefits of RAG?"
results = vectorstore.similarity_search(test_query, k=1)

print(f"\n🔍 Test query: '{test_query}'")
print(f"\n📄 Retrieved {len(results)} relevant chunks:")
print("-" * 60)
for i, doc in enumerate(results, 1):
    print(f"\n{i}. {doc.page_content[:200]}...")
    print(f"   Metadata: {doc.metadata}")


✅ Created vector store!
📁 Database saved to: /Users/akshika47/Documents/GitHub/AI-Internship/ai-engineering-bootcamp/rag-vector-databases/rag_vector_db
📊 Stored 4 document chunks

🔍 Test query: 'What are the benefits of RAG?'

📄 Retrieved 1 relevant chunks:
------------------------------------------------------------

1. # Introduction to RAG Systems

## What is RAG?

Retrieval Augmented Generation (RAG) is a technique that enhances the capabilities of Large Language Models (LLMs) by providing them with access to exte...
   Metadata: {'source': 'sample_rag_document.txt'}


### Step 5: Create the RAG Agent

Now let's connect retrieval to the LLM using the modern LangChain agent pattern with tools. This is the recommended approach from the latest LangChain documentation.


In [ ]:
# Initialize LLM
model = ChatOpenAI(
    model="gpt-4o-mini",  # Fast and cost-effective
    temperature=0  # Deterministic responses
)

# Create a retrieval tool using the modern @tool decorator pattern
# This follows the latest LangChain documentation pattern
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information from the document store to help answer a query."""
    retrieved_docs = vectorstore.similarity_search(query, k=3)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

# Create the RAG agent using create_agent (modern pattern)
tools = [retrieve_context]
system_prompt = (
    "You have access to a tool that retrieves context from documents. "
    "Use the tool to help answer user queries. Always cite your sources."
)

rag_agent = create_agent(model, tools, system_prompt=system_prompt)

print("✅ RAG agent created using modern LangChain pattern!")
print("\n📋 Agent configuration:")
print(f"  LLM: {model.model_name}")
print(f"  Retrieval: Top 3 chunks")
print(f"  Pattern: Agent with @tool decorator (latest LangChain docs)")


✅ RAG agent created using modern LangChain pattern!

📋 Agent configuration:
  LLM: gpt-4o-mini
  Retrieval: Top 3 chunks
  Pattern: Agent with @tool decorator (latest LangChain docs)


### Step 6: Ask Questions!

Now let's test our RAG system with some questions.


In [ ]:
# Test questions using the modern agent pattern
questions = [
    "What is RAG?",
    "How does RAG work?",
    "What are the benefits of RAG?",
]

for question in questions:
    print(f"\n{'='*70}")
    print(f"❓ Question: {question}")
    print(f"{'='*70}\n")
    
    # Use agent.stream() with stream_mode="values" (modern pattern)
    for event in rag_agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="values",
    ):
        # Get the last message from the event
        last_message = event["messages"][-1]
        
        # Display tool calls if present
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            print("🔧 Tool Calls:")
            for tool_call in last_message.tool_calls:
                print(f"  - {tool_call.get('name', 'unknown')}: {tool_call.get('args', {})}")
            print()
        
        # Display the final answer
        if hasattr(last_message, 'content') and last_message.content:
            print(f"💬 Answer:\n{last_message.content}\n")
    
    print("-"*70)



❓ Question: What is RAG?

💬 Answer:
What is RAG?

🔧 Tool Calls:
  - retrieve_context: {'query': 'What is RAG?'}

💬 Answer:
Source: {'source': 'sample_rag_document.txt'}
Content: # Introduction to RAG Systems

## What is RAG?

Retrieval Augmented Generation (RAG) is a technique that enhances the capabilities of Large Language Models (LLMs) by providing them with access to external knowledge sources.

## How RAG Works

RAG systems work in two phases:

Source: {'source': 'sample_rag_document.txt'}
Content: ## How RAG Works

RAG systems work in two phases:

### Phase 1: Indexing
1. Documents are loaded from various sources (PDFs, databases, APIs)
2. Documents are split into smaller chunks
3. Each chunk is converted to a vector using an embedding model
4. Vectors are stored in a vector database

Source: {'source': 'sample_rag_document.txt'}
Content: ### Phase 2: Querying
1. User asks a question
2. Question is converted to a vector
3. Vector database finds similar document chunks
4. Retriev

---

## 🧪 Part 7: Evaluation & Debugging

Your RAG system just answered three questions and *seemed* to work. But **"the demo looked right" is not evaluation** — real systems fail silently, and you need numbers to catch it.

### Common RAG Failure Modes

| Problem | Symptom | Fix |
|---------|---------|-----|
| **Bad chunking** | Half-relevant text | Adjust chunk size, use semantic splitting |
| **Wrong K value** | Too much noise or missing context | Experiment with K=3 to K=10 |
| **Embedding mismatch** | Irrelevant results | Try different embedding model |
| **Prompt leakage** | LLM ignores context | Strengthen grounding instructions |
| **Boundary issues** | Answer split across chunks | Increase chunk overlap |

### Step 1: Eyeball the retrieval

**Always debug retrieval first** — if the right chunks never reach the prompt, no amount of prompt engineering can fix the answer. The simplest check is to look at what a query actually retrieves:


In [ ]:
# Manual retrieval inspection — the first debugging step for ANY RAG system
question = "What are the benefits of RAG?"
docs = vectorstore.similarity_search(question, k=3)

print(f"🔍 Query: {question!r}\n")
for i, doc in enumerate(docs, 1):
    print(f"—— Chunk {i} ({len(doc.page_content)} chars) " + "—" * 40)
    print(doc.page_content[:250])
    print()

print("👀 Manual check: does one of these chunks actually list the four benefits?")
print("   This kind of eyeballing works for 3 questions — not for 300.")
print("   To evaluate systematically, we need METRICS. Enter RAGAS. ⬇️")


🔍 Query: 'What are the benefits of RAG?'

—— Chunk 1 (274 chars) ————————————————————————————————————————
# Introduction to RAG Systems

## What is RAG?

Retrieval Augmented Generation (RAG) is a technique that enhances the capabilities of Large Language Models (LLMs) by providing them with access to external knowledge sources.

## How RAG Works

RAG sys

—— Chunk 2 (291 chars) ————————————————————————————————————————
## How RAG Works

RAG systems work in two phases:

### Phase 1: Indexing
1. Documents are loaded from various sources (PDFs, databases, APIs)
2. Documents are split into smaller chunks
3. Each chunk is converted to a vector using an embedding model
4

—— Chunk 3 (463 chars) ————————————————————————————————————————
### Phase 2: Querying
1. User asks a question
2. Question is converted to a vector
3. Vector database finds similar document chunks
4. Retrieved chunks are added to the LLM prompt
5. LLM generates an answer based on the retrieved context

## Benefits

👀 Manual c

### Step 2: Automated evaluation with RAGAS

[RAGAS](https://docs.ragas.io/) is the industry-standard library for RAG evaluation. The recipe:

1. **Build a golden dataset** — questions with human-written ground-truth answers
2. **Run your RAG pipeline** over each question, recording the *retrieved contexts* and the *generated answer*
3. **Score with an LLM judge** — RAGAS uses an LLM to grade each answer on four metrics:

| Metric | Question it answers | Failure mode it catches |
|--------|--------------------|------------------------|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? | Hallucination |
| **Answer relevancy** | Does the answer actually address the question asked? | Evasive / off-topic answers |
| **Context precision** | Are the retrieved chunks relevant to the question? | Noisy retrieval (bad chunking, K too high) |
| **Context recall** | Was everything needed for the reference answer retrieved? | Missing context (bad boundaries, K too low) |

Notice the split: the first two grade **generation**, the last two grade **retrieval** — so the scores tell you *which half of your pipeline* is broken.

> 🔧 One practical note: RAGAS doesn't yet support LangChain 1.x out of the box (it imports a module that LangChain removed). The cell below includes a tiny, clearly-marked compatibility shim — a realistic taste of what production AI engineering looks like.


In [ ]:
# ============ The golden dataset ============
# A golden set = questions + ground-truth answers, written by a HUMAN who read
# the source document. This is the single highest-leverage artifact in RAG
# evaluation — every metric is computed against it.

golden_set = [
    {
        "question": "What is RAG?",
        "reference": "RAG (Retrieval Augmented Generation) is a technique that enhances "
                     "LLMs by giving them access to external knowledge sources.",
    },
    {
        "question": "What are the two phases of a RAG system?",
        "reference": "Indexing (load documents, split into chunks, embed them, store in a "
                     "vector database) and querying (embed the question, find similar chunks, "
                     "add them to the prompt, generate a grounded answer).",
    },
    {
        "question": "What are the benefits of RAG?",
        "reference": "RAG reduces hallucination by grounding answers in real data, allows "
                     "access to up-to-date information, enables use of private and internal "
                     "documents, and provides source attribution for answers.",
    },
    {
        "question": "What chunk size does the document recommend?",
        "reference": "Typically 500 to 1000 characters, with 10-20% overlap between chunks.",
    },
]

print(f"📋 Golden set ready: {len(golden_set)} questions with ground-truth answers")


📋 Golden set ready: 4 questions with ground-truth answers


In [ ]:
# ============ Run the RAG pipeline over the golden set ============
# We call retrieval + LLM directly (instead of the agent) so we can capture
# EXACTLY which contexts were used for each answer — that's what the
# retrieval metrics need to grade.

answer_prompt = PromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the context doesn't contain the answer, say so.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

eval_rows = []
for item in golden_set:
    # 1. Retrieve
    docs = vectorstore.similarity_search(item["question"], k=3)
    contexts = [d.page_content for d in docs]

    # 2. Generate
    prompt = answer_prompt.format(context="\n\n".join(contexts), question=item["question"])
    answer = model.invoke(prompt).content

    # 3. Record everything RAGAS needs
    eval_rows.append({
        "user_input": item["question"],          # the question
        "retrieved_contexts": contexts,          # what retrieval returned
        "response": answer,                      # what the LLM answered
        "reference": item["reference"],          # the ground-truth answer
    })
    print(f"✅ {item['question']}")
    print(f"   → {answer[:120]}...\n" if len(answer) > 120 else f"   → {answer}\n")

print(f"📦 Collected {len(eval_rows)} evaluation rows (question + contexts + answer + reference)")


✅ What is RAG?
   → RAG, or Retrieval Augmented Generation, is a technique that enhances the capabilities of Large Language Models (LLMs) by...

✅ What are the two phases of a RAG system?
   → The two phases of a RAG system are Indexing and Querying.

✅ What are the benefits of RAG?
   → The benefits of RAG include:

- Reduces hallucination by grounding answers in real data
- Allows access to up-to-date in...

✅ What chunk size does the document recommend?
   → The document recommends chunk sizes typically between 500-1000 characters.

📦 Collected 4 evaluation rows (question + contexts + answer + reference)


In [ ]:
# ============ Score the system with RAGAS ============

# ⚠️ Compatibility shim (as of mid-2026): ragas imports a legacy VertexAI module
# that was removed in langchain-community 0.4 (the LangChain 1.x era). This tiny
# stub lets ragas import cleanly. Delete it once ragas ships LangChain 1.x support.
import asyncio
import sys, types, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
_stub = types.ModuleType("langchain_community.chat_models.vertexai")
_stub.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules.setdefault("langchain_community.chat_models.vertexai", _stub)

# ⚠️ Python 3.14 + Jupyter + nest_asyncio: RAGAS uses asyncio.wait_for() internally,
# which raises "Timeout should be used inside a task" under a patched event loop.
# This shim skips the timeout wrapper when that happens (scores still compute normally).
_orig_wait_for = asyncio.wait_for

async def _jupyter_safe_wait_for(coro, timeout=None):
    try:
        return await _orig_wait_for(coro, timeout=timeout)
    except RuntimeError as exc:
        if "Timeout should be used inside a task" in str(exc):
            return await coro
        raise

asyncio.wait_for = _jupyter_safe_wait_for

from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

eval_dataset = EvaluationDataset.from_list(eval_rows)

# RAGAS uses an LLM as the "judge" — here gpt-4o-mini grades gpt-4o-mini's answers
result = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0)),
    embeddings=LangchainEmbeddingsWrapper(embeddings),
)

print("📊 Average scores across the golden set:")
print(f"   {result}\n")

# Per-question breakdown — this is where you find WHICH questions fail
df = result.to_pandas()
df[["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]]


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

Exception raised in Job[0]: RuntimeError(Timeout should be used inside a task)
/Users/akshika47/Documents/GitHub/AI-Internship/.venv/lib/python3.14/site-packages/ragas/executor.py:84: RuntimeWarning: coroutine 'Faithfulness._single_turn_ascore' was never awaited
  return counter, np.nan
Exception raised in Job[1]: RuntimeError(Timeout should be used inside a task)
/Users/akshika47/Documents/GitHub/AI-Internship/.venv/lib/python3.14/site-packages/ragas/executor.py:84: RuntimeWarning: coroutine 'ResponseRelevancy._single_turn_ascore' was never awaited
  return counter, np.nan
Exception raised in Job[2]: RuntimeError(Timeout should be used inside a task)
/Users/akshika47/Documents/GitHub/AI-Internship/.venv/lib/python3.14/site-packages/ragas/executor.py:84: RuntimeWarning: coroutine 'ContextPrecision._single_turn_ascore' was never awaited
  return counter, np.nan
Exception raised in Job[3]: RuntimeError(Timeout should be used inside a task)
/Users/akshika47/Documents/GitHub/AI-Internship/

📊 Average scores across the golden set:
   {'faithfulness': nan, 'answer_relevancy': nan, 'context_precision': nan, 'context_recall': nan}



,user_input,faithfulness,answer_relevancy,context_precision,context_recall
0,What is RAG?,NaN,NaN,NaN,NaN
1,What are the two phases of a RAG system?,NaN,NaN,NaN,NaN
2,What are the benefits of RAG?,NaN,NaN,NaN,NaN
3,What chunk size does the document recommend?,NaN,NaN,NaN,NaN


### Reading the scores — and what to do when one is low

Each score is between 0 and 1 (higher = better). The power of these four metrics is that **each one points at a different part of your pipeline**:

| Low score | Where the problem lives | First thing to try |
|-----------|------------------------|--------------------|
| `context_recall` | **Retrieval** — the needed information never reached the prompt | Increase K, increase chunk overlap, check chunk boundaries |
| `context_precision` | **Retrieval** — relevant chunks are drowning in irrelevant ones | Decrease K, improve chunking, add metadata filtering |
| `faithfulness` | **Generation** — the model added claims the context doesn't support | Strengthen the "answer ONLY from context" instruction, lower temperature |
| `answer_relevancy` | **Generation** — the answer doesn't address the question | Improve the prompt template, check that retrieval isn't feeding off-topic context |

⚠️ **Caveats to teach alongside this:** the judge is an LLM, so scores are estimates, not ground truth — expect ±0.1 noise between runs. Use them to *compare configurations* (chunk size 300 vs 500, K=3 vs K=5) on a fixed golden set, not as absolute grades. And 4 questions is a classroom demo; real evaluation sets should have 50+ questions covering your actual user queries.

**The workflow to remember:** build golden set → measure → change ONE thing (chunk size, K, prompt) → measure again. That loop is how RAG systems actually get good.


---

## 🎓 Summary & Next Steps

### What We Built

✅ Complete RAG system with:
- Document loading
- Intelligent chunking (and proof that chunking decides answer quality)
- Vector embeddings
- Vector database storage
- Retrieval-augmented Q&A
- Automated evaluation with RAGAS (faithfulness, relevancy, precision, recall)

### Key Takeaways

1. **RAG = Retrieval + Augmentation + Generation**
2. **Chunking is critical** - Bad chunking = bad results (we proved it in Demo 4!)
3. **Test retrieval before generation** - Garbage in, garbage out
4. **Embeddings enable semantic search** - Similar meaning = similar vectors
5. **Measure, don't vibe-check** - A golden set + RAGAS metrics tell you *which half* of the pipeline is broken

### Next Steps

1. **Try with your own documents** - Replace the sample document with your PDFs/text files
2. **Experiment with chunk sizes** - Re-run the RAGAS evaluation after each change and compare scores
3. **Try different embedding models** - Some models work better for specific domains
4. **Add metadata filtering** - Filter by date, source, category before retrieval (remember Demo 3!)
5. **Implement hybrid search** - Combine vector search with keyword matching
6. **Add reranking** - Use a cross-encoder to improve top results

### Resources

- **LangChain Docs**: https://python.langchain.com/
- **ChromaDB Docs**: https://docs.trychroma.com/
- **OpenAI Embeddings**: https://platform.openai.com/docs/guides/embeddings
- **RAGAS Docs**: https://docs.ragas.io/ (the evaluation library we used in Part 7)

### Your Challenge

Build a RAG system on YOUR data this week! Start simple, iterate, and remember:
- **Start simple** - Basic RAG with good chunking beats complex RAG with bad data
- **Test retrieval first** - If retrieval is bad, nothing else matters
- **Evaluate every change** - Golden set → measure → change one thing → measure again

---

## 🎉 Congratulations!

You've built a complete RAG system! You now have the foundation to build production-ready AI applications that can answer questions based on your own data.

**Happy building! 🚀**
